# Engram: final test on conversations 5 to 9

The design fixed in frozen_design.json, on conversations it has never seen. This fair test can be used only once. Networks train on conversations 0 to 4. Benchmark data only.

**Kaggle:** Settings > Accelerator > GPU T4 x2, and Settings > Turn on internet. Then Run > Run all. Keep the tab open.

In [ ]:
COMMIT = "34731b1"
import os, subprocess, time, urllib.request
gpu = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.strip()
if not gpu: raise SystemExit("No GPU. Choose Settings > Accelerator > GPU T4 x2.")
print("GPU:", gpu.replace(chr(10), ", "))
try:
    urllib.request.urlopen("https://github.com", timeout=10)
except Exception:
    raise SystemExit("No internet. Choose Settings > Turn on internet, then run again.")
WORK = "/kaggle/working"
os.chdir(WORK)
subprocess.run("rm -rf Engram; git clone -q https://github.com/manjunathhari/Engram.git", shell=True, check=True)
os.chdir(f"{WORK}/Engram")
subprocess.run(f"git checkout -q {COMMIT}; git clone -q --depth 1 https://github.com/snap-research/locomo.git", shell=True, check=True)
subprocess.run("pip install -q wordllama scikit-learn spacy; python -m spacy download en_core_web_sm", shell=True, check=True, capture_output=True)
import wordllama, shutil
p = os.path.dirname(wordllama.__file__)
a, b = os.path.join(p, "tokenizers"), os.path.join(p, "tokenizer")
if os.path.isdir(a) and not os.path.exists(b): shutil.copytree(a, b)
subprocess.run("apt-get -qq install -y zstd pciutils > /dev/null; curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1", shell=True)
env = dict(os.environ, OLLAMA_CONTEXT_LENGTH="4096", OLLAMA_KEEP_ALIVE="-1")
subprocess.Popen(["ollama", "serve"], env=env, stdout=open(f"{WORK}/ollama.log", "a"), stderr=subprocess.STDOUT, start_new_session=True)
time.sleep(5)
MODEL = "hf.co/bartowski/google_gemma-3-4b-it-GGUF:Q4_K_M"
r = subprocess.run(["ollama", "pull", MODEL], capture_output=True, text=True)
if r.returncode: raise SystemExit("Model download failed: " + r.stderr[-200:])
subprocess.run(["ollama", "cp", MODEL, "gemma3:4b"])
print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)
print("setup ready")

In [ ]:
%%writefile /kaggle/working/final_test.py
"""Final test of the design in frozen_design.json on conversations 5-9. Benchmark data only.
Networks train on conversations 0-4. Three ways answer the same questions: networks with facts, networks without facts, and current Engram."""
import argparse, json, os, sys, tempfile, time
sys.path.insert(0, os.getcwd())
import vps_kit as K
import engram_chat as E
from engram import Config
import engram_parallel as P

ap = argparse.ArgumentParser()
ap.add_argument("--url", required=True); ap.add_argument("--model", required=True); ap.add_argument("--save", required=True)
ap.add_argument("--baseline", required=True); ap.add_argument("--sample", type=int, default=100000)
a = ap.parse_args()
TRAIN, TEST = [0, 1, 2, 3, 4], [5, 6, 7, 8, 9]
D = json.load(open("frozen_design.json"))
assert D["sentence_tokens"] == 800 and D["fact_tokens"] == 300
tools = P._load_tools()
cfg = Config(**json.load(open("final_config.json")))
tmp = tempfile.mkdtemp()
convs = {c: K.load_conv(K.DATA[c]) for c in TRAIN + TEST}
llm = K.LLM(a.url, a.model, "local", 1200)
cache_path = f"{a.save}/facts_final.json"
cache = json.load(open(cache_path)) if os.path.exists(cache_path) else {}


def replay(c, search, use_helper=False):
    state = {}

    def helper(prompt, max_tokens=0):
        key = state["key"]
        if key not in cache:
            t0 = time.time()
            r = llm(prompt, max_tokens=max_tokens)[0]
            cache[key] = dict(raw=r, secs=round(time.time() - t0, 1))
            json.dump(cache, open(cache_path, "w"))
            print("fact file", key, flush=True)
        return cache[key]["raw"], {}, cache[key]["secs"]
    m = E.ChatMemory(f"{tmp}/{c}_{id(state)}.db", cfg=cfg, durable=False, search=search, helper=helper if use_helper else None, helper_name=a.model if use_helper else "")
    last = None
    for t in convs[c].turns:
        if t["session"] != last:
            if last is not None:
                state["key"] = f"{c}:{last}"; m.close_chat(write_note=False, sleep=False)
            m.open_chat(t["date"]); last = t["session"]
        m.add(t["speaker"], t["text"], date=t["date"], ref=t["id"], caption=t["cap"], commit=False)
    m.db.commit(); state["key"] = f"{c}:{last}"; m.close_chat(write_note=False)
    return m


print("building training memories", flush=True)
TR = {c: replay(c, P.ChannelSearch("plain", tools=tools)) for c in TRAIN}
ref = {c: dict(TR[c].db.execute("SELECT ref, sid FROM sentences").fetchall()) for c in TRAIN}
ex = {c: [(TR[c].search, q["q"], {ref[c][e] for e in q["ev"] if e in ref[c]}) for q in convs[c].qs if q["cat"] != 5 and q["ev"]] for c in TRAIN}
print("training networks", flush=True)
everything = [x for c in TRAIN for x in ex[c] if x[2]]
NET = [P.train(everything, s) for s in (1, 2, 3)]

# Check first, on development questions only, that this is the same model and answer path as the earlier runs.
base = {(r["conv"], r["i"]): r for r in map(json.loads, open(a.baseline)) if r["system"] == "network"}
held = {}
for c in TRAIN:
    TR[c].search.mode = "network"
    TR[c].search.model = [P.train([x for d in TRAIN if d != c for x in ex[d] if x[2]], s) for s in (1, 2, 3)]
same_ev = same_ans = checks = 0
dev = [(c, i) for c, i in K.balanced({c: convs[c] for c in TRAIN}, TRAIN, 240) if convs[c].qs[i]["cat"] != 5][:20]
for c, i in dev:
    b = base.get((c, i))
    if not b: continue
    q = convs[c].qs[i]
    ctx, info = TR[c].context_for(q["q"])
    gold = {ref[c][e] for e in q["ev"] if e in ref[c]}
    rec = len(set(info["sids"]) & gold) / len(gold)
    pred = llm(K.ANSWER.format(ctx=ctx, q=q["q"]), max_tokens=60)[0]
    checks += 1; same_ev += abs(rec - b["recall"]) < 1e-9; same_ans += pred.strip() == b["pred"].strip()
print(f"check on development questions: evidence same on {same_ev}/{checks}, answers word for word on {same_ans}/{checks}", flush=True)
if checks and same_ev < checks * .7:
    print("STOPPED: search does not match the earlier run. Send this to Claude.", flush=True)
    sys.exit(3)

print("writing facts for the test conversations", flush=True)
MEM = {c: replay(c, P.ChannelSearch("network", model=NET, tools=tools), use_helper=True) for c in TEST}
CUR = {c: replay(c, None) for c in TEST}
print("facts ready:", sum(m.db.execute("SELECT count(*) FROM facts").fetchone()[0] for m in MEM.values()), "from", len(cache), "chats", flush=True)
REF = {c: dict(MEM[c].db.execute("SELECT ref, sid FROM sentences").fetchall()) for c in TEST}
plan = [(c, i) for c, i in K.balanced({c: convs[c] for c in TEST}, TEST, a.sample) if convs[c].qs[i]["cat"] != 5]
plan = [(c, i) for c, i in plan if any(e in REF[c] for e in convs[c].qs[i]["ev"])]
print(len(plan), "questions, 3 ways each", flush=True)
path = f"{a.save}/final_answers.jsonl"
done = {(r["conv"], r["i"], r["system"]) for r in map(json.loads, open(path))} if os.path.exists(path) else set()
with open(path, "a") as out:
    for n, (c, i) in enumerate(plan, 1):
        q = convs[c].qs[i]
        gold = {REF[c][e] for e in q["ev"] if e in REF[c]}
        for name in ("with_facts", "no_facts", "current"):
            if (c, i, name) in done: continue
            m = CUR[c] if name == "current" else MEM[c]
            m.use_facts = name == "with_facts"
            ctx, info = m.context_for(q["q"])
            pred, _, secs = llm(K.ANSWER.format(ctx=ctx, q=q["q"]), max_tokens=60)
            out.write(json.dumps(dict(conv=c, i=i, system=name, cat=K.CATS[q["cat"]], q=q["q"], gold=q["answer"], pred=pred, recall=len(set(info["sids"]) & gold) / len(gold), tokens=info["tokens"], secs=secs)) + "\n")
            out.flush()
        if n % 25 == 0: print(f"question {n}/{len(plan)}", flush=True)
print("Done", flush=True)

In [ ]:
import os, subprocess, sys, time
WORK, SAVE = "/kaggle/working", "/kaggle/working/engram_results"
os.makedirs(SAVE, exist_ok=True); os.chdir(f"{WORK}/Engram")
LOG = f"{SAVE}/final_run.log"
if subprocess.run(["pgrep", "-f", "final_test.py"], capture_output=True).returncode != 0:
    subprocess.Popen([sys.executable, "-u", f"{WORK}/final_test.py", "--url", "http://localhost:11434/v1", "--model", "gemma3:4b", "--save", SAVE, "--baseline", "results/helper_hf_dev.jsonl"], stdout=open(LOG, "a"), stderr=subprocess.STDOUT, start_new_session=True)
    print("test started")
for _ in range(960):
    time.sleep(15)
    t = open(LOG).read().splitlines()
    print(chr(13) + " | ".join(x for x in t if not x.startswith("fact file"))[-150:], end="")
    if t and (t[-1].startswith("Done") or "STOPPED" in t[-1] or "Traceback" in "\n".join(t[-30:])): break
print(); print("\n".join(open(LOG).read().splitlines()[-25:])); print(sorted(os.listdir(SAVE)))

When it says Done, send Claude `final_answers.jsonl`, `facts_final.json` and `final_run.log` from the `engram_results` folder in the Output panel.